![image_1780163880671.png](./image_1780163880671.png "image_1780163880671.png")

![image_1780163893050.png](./image_1780163893050.png "image_1780163893050.png")

In [0]:
from pyspark.sql import SparkSession

# Initialize Spark
spark = SparkSession.builder.appName("OrdersData").getOrCreate()

# Create DataFrame
orders_data = [
    (1, 101, "2023-01-05"),
    (2, 102, "2023-01-18"),
    (3, 103, "2023-02-02"),
    (4, 101, "2023-02-14"),
    (5, 104, "2023-02-20"),
    (6, 105, "2023-03-01"),
    (7, 102, "2023-03-10"),
    (8, 106, "2023-03-15"),
    (9, 107, "2023-03-22"),
    (10, 105, "2023-04-05"),
]

orders_df = spark.createDataFrame(orders_data, ["id", "customer_id", "order_date"])

# Show DataFrame
orders_df.show()


In [0]:
from pyspark.sql import functions as f
from pyspark.sql import Window

window = Window.partitionBy("customer_id").orderBy("order_date")

result_df = (
    orders_df.withColumn("rank", f.dense_rank().over(window))
    .withColumn("year_month", f.date_format(f.col("order_date"), "yyyy-MM"))
    .filter(f.col("rank") == 1)
    .groupBy(f.col("year_month"))
    .agg(f.count("*").alias("new_customers"))
    .select(f.col("year_month"), f.col("new_customers"))
    .orderBy(f.col("year_month"))
)
result_df.show()